# Notebook 02: Near-Duplicate Audit and Reproducible Data Splits

This notebook:

1. Reviews cross-split perceptual-hash near-duplicate candidates.
2. Excludes only confirmed cross-split leakage cases.
3. Creates a reproducible 85/15 train/calibration split from the original training partition.
4. Locks the cleaned original validation partition as the untouched internal test set.
5. Exports separate manifests for:
   - Main 19-class disease classification task
   - Auxiliary 20-class classification task including healthy soybean

In [ ]:
!pip install -q imagehash scikit-learn

import os
import json
import random
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 7

random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

OUTPUT_DIR = WORK_DIR / "split_outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MANIFEST_DIR = OUTPUT_DIR / "manifests"
METADATA_DIR = OUTPUT_DIR / "metadata"

for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MANIFEST_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PLANTVILLAGE_ROOT = Path(
    "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage"
)

assert PLANTVILLAGE_ROOT.exists(), (
    f"PlantVillage root not found:\n{PLANTVILLAGE_ROOT}"
)

print("PlantVillage root:")
print(PLANTVILLAGE_ROOT)

print("\nOutput directory:")
print(OUTPUT_DIR)

In [ ]:
def find_file_under_kaggle_input(file_name):
    matches = list(INPUT_ROOT.rglob(file_name))
    
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not locate '{file_name}' under /kaggle/input.\n"
            "Attach the output dataset from Notebook 01 to this notebook."
        )
    
    return matches[0]


CLEAN_MANIFEST_PATH = find_file_under_kaggle_input(
    "plantvillage_selected_manifest_clean.csv"
)

NEAR_DUPLICATE_PATH = find_file_under_kaggle_input(
    "near_duplicate_candidates_phash_hamming_le4.csv"
)

print("Clean manifest:")
print(CLEAN_MANIFEST_PATH)

print("\nNear-duplicate candidates:")
print(NEAR_DUPLICATE_PATH)

In [ ]:
master_df = pd.read_csv(CLEAN_MANIFEST_PATH)
near_df = pd.read_csv(NEAR_DUPLICATE_PATH)

print("Master manifest shape:", master_df.shape)
print("Near-duplicate candidate shape:", near_df.shape)

print("\nMaster columns:")
print(master_df.columns.tolist())

print("\nNear-duplicate candidate columns:")
print(near_df.columns.tolist())

required_master_columns = {
    "image_path", "split", "crop", "disease",
    "label", "is_healthy", "sha256", "phash"
}

missing_master = required_master_columns - set(master_df.columns)

assert not missing_master, (
    f"Missing required columns in clean manifest: {missing_master}"
)

assert master_df["split"].isin(["train", "val"]).all(), (
    "Master manifest should contain only original train and val records."
)

assert master_df["is_corrupt"].sum() == 0, (
    "Clean manifest unexpectedly contains corrupt images."
)

assert master_df["label"].nunique() == 20, (
    "Expected 20 labels in the full selected subset."
)

print("\nValidation passed.")
print("Clean images:", len(master_df))
print("Classes:", master_df["label"].nunique())

In [ ]:
display(near_df.head())

print("\nCandidate count:", len(near_df))

print("\nCandidate distance distribution:")
distance_columns = [
    col for col in near_df.columns
    if "distance" in col.lower() or "hamming" in col.lower()
]

print(distance_columns)

if distance_columns:
    display(
        near_df[distance_columns]
        .describe()
        .T
    )

In [ ]:
def find_first_matching_column(columns, keywords):
    for column in columns:
        column_lower = column.lower()
        if all(keyword in column_lower for keyword in keywords):
            return column
    return None


candidate_columns = near_df.columns.tolist()

path_a_col = (
    find_first_matching_column(candidate_columns, ["path", "a"]) or
    find_first_matching_column(candidate_columns, ["image", "a"]) or
    find_first_matching_column(candidate_columns, ["path", "1"]) or
    find_first_matching_column(candidate_columns, ["image", "1"])
)

path_b_col = (
    find_first_matching_column(candidate_columns, ["path", "b"]) or
    find_first_matching_column(candidate_columns, ["image", "b"]) or
    find_first_matching_column(candidate_columns, ["path", "2"]) or
    find_first_matching_column(candidate_columns, ["image", "2"])
)

distance_col = (
    find_first_matching_column(candidate_columns, ["hamming"]) or
    find_first_matching_column(candidate_columns, ["distance"])
)

print("Detected image A path column:", path_a_col)
print("Detected image B path column:", path_b_col)
print("Detected distance column:", distance_col)

assert path_a_col is not None, "Could not identify first image-path column."
assert path_b_col is not None, "Could not identify second image-path column."
assert distance_col is not None, "Could not identify Hamming-distance column."

review_df = pd.DataFrame({
    "candidate_id": np.arange(1, len(near_df) + 1),
    "image_a_path": near_df[path_a_col].astype(str),
    "image_b_path": near_df[path_b_col].astype(str),
    "phash_hamming_distance": near_df[distance_col],
})

review_df.head()

In [ ]:
metadata_lookup = (
    master_df[
        [
            "image_path", "split", "crop", "disease",
            "label", "file_name", "sha256", "phash"
        ]
    ]
    .drop_duplicates("image_path")
    .copy()
)

left_metadata = metadata_lookup.rename(
    columns={
        "image_path": "image_a_path",
        "split": "split_a",
        "crop": "crop_a",
        "disease": "disease_a",
        "label": "label_a",
        "file_name": "file_name_a",
        "sha256": "sha256_a",
        "phash": "phash_a"
    }
)

right_metadata = metadata_lookup.rename(
    columns={
        "image_path": "image_b_path",
        "split": "split_b",
        "crop": "crop_b",
        "disease": "disease_b",
        "label": "label_b",
        "file_name": "file_name_b",
        "sha256": "sha256_b",
        "phash": "phash_b"
    }
)

review_df = review_df.merge(left_metadata, on="image_a_path", how="left")
review_df = review_df.merge(right_metadata, on="image_b_path", how="left")

review_df["is_cross_split"] = (
    review_df["split_a"].notna() &
    review_df["split_b"].notna() &
    (review_df["split_a"] != review_df["split_b"])
)

review_df["same_label"] = (
    review_df["label_a"].notna() &
    review_df["label_b"].notna() &
    (review_df["label_a"] == review_df["label_b"])
)

review_df["same_sha256"] = (
    review_df["sha256_a"].notna() &
    review_df["sha256_b"].notna() &
    (review_df["sha256_a"] == review_df["sha256_b"])
)

review_df["review_decision"] = "PENDING"
review_df["review_reason"] = ""
review_df["exclude_image_path"] = ""
review_df["reviewer"] = ""
review_df["review_date"] = ""

cross_split_review_df = review_df[
    review_df["is_cross_split"]
].copy()

print("All pHash candidates:", len(review_df))
print("Cross-split pHash candidates:", len(cross_split_review_df))
print("Same-label cross-split candidates:", cross_split_review_df["same_label"].sum())
print("Different-label cross-split candidates:", (~cross_split_review_df["same_label"]).sum())

display(
    cross_split_review_df[
        [
            "candidate_id",
            "phash_hamming_distance",
            "split_a", "label_a",
            "split_b", "label_b",
            "same_label",
            "same_sha256"
        ]
    ].head(20)
)

In [ ]:
review_sheet_path = TABLE_DIR / "near_duplicate_review_decisions.csv"

cross_split_review_df.to_csv(review_sheet_path, index=False)

print("Saved review sheet:")
print(review_sheet_path)

print("\nImportant:")
print("Do not edit the original Kaggle dataset files.")
print("Review the visual sheets, then edit only review_decision, review_reason,")
print("exclude_image_path, reviewer, and review_date in this CSV.")